# Data Acquisition and Quality Control

## Overview
This notebook downloads and validates <em>Mycobacterium tuberculosis</em> genome data for comparative genomics analysis between strains or variants of the bacterium.

In [1]:
# Import libraries and modules for setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Set project root directory
project_root = Path.cwd().parent
scripts_path = project_root / 'Scripts'
sys.path.append(str(scripts_path))

In [3]:
# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Verify project root path
print(f"Project root: {project_root}")
print(f"Current working directory: {Path.cwd()}")
print(f"Scripts directory exists: {scripts_path.exists()}")

Project root: C:\Users\dalto\Desktop\Education Resources\Personal\Projects\Mycobacterium_tuberculosis_Comparative_Genomics
Current working directory: C:\Users\dalto\Desktop\Education Resources\Personal\Projects\Mycobacterium_tuberculosis_Comparative_Genomics\Notebooks
Scripts directory exists: True


In [4]:
# Validate Scripts directory contents
print("Scripts directory contents:")
for file in scripts_path.glob("*.py"):
    print(f" - {file.name}")

# Check system Path
print("\\nFirst 5 entries in sys.path:")
for path in sys.path[:5]:
    print(f" - {path}")

# Try importing using absolute path
print("\\nTrying alternative methods...")
try:
    import importlib.util

    # Try importing download_data
    download_data_path = scripts_path / 'download_data.py'
    if download_data_path.exists():
        spec = importlib.util.spec_from_file_location("download_data", download_data_path)
        download_data_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(download_data_module)
        MTBDataDownloader = download_data_module.MTBDataDownloader
        print("✓ download_data imported via spec")
    else:
        print("✗ download_data.py not found")
    
    # Try importing genome_analyzer
    genome_analyzer_path = scripts_path / 'genome_analyzer.py'
    if genome_analyzer_path.exists():
        spec = importlib.util.spec_from_file_location("genome_analyzer", genome_analyzer_path)
        genome_analyzer_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(genome_analyzer_module)
        GenomeAnalyzer = genome_analyzer_module.GenomeAnalyzer
        print("✓ genome_analyzer imported via spec")
    else:
        print("✗ genome_analyzer.py not found")
except Exception as e:
    print(f"Error with alternative import method: {e}")

print("\\nImport Status:")
if 'MTBDataDownloader' in locals():
    print("✓ MTBDataDownloader is available")
else:
    print("✗ MTBDataDownloader is NOT available")
    
if 'GenomeAnalyzer' in locals():
    print("✓ GenomeAnalyzer is available")
else:
    print("✗ GenomeAnalyzer is NOT available")

Scripts directory contents:
 - alignment_analysis.py
 - annotation_parser.py
 - download_data.py
 - genome_analyzer.py
 - integrated_analysis.py
\nFirst 5 entries in sys.path:
 - C:\Users\dalto\anaconda3\envs\mtb_comp_genomics\python314.zip
 - C:\Users\dalto\anaconda3\envs\mtb_comp_genomics\DLLs
 - C:\Users\dalto\anaconda3\envs\mtb_comp_genomics\Lib
 - C:\Users\dalto\anaconda3\envs\mtb_comp_genomics
 - 
\nTrying alternative methods...
Error with alternative import method: cannot import name 'GC' from 'Bio.SeqUtils' (C:\Users\dalto\anaconda3\envs\mtb_comp_genomics\Lib\site-packages\Bio\SeqUtils\__init__.py)
\nImport Status:
✗ MTBDataDownloader is NOT available
✗ GenomeAnalyzer is NOT available


In [ ]:
try:
    from download_data import MTBDataDownloader
    from genome_analyzer import GenomeAnalyzer
    print("✓ Custom modules imported successfully.")
except ImportError as e:
    print("Error importing modules: {e}.")
    print("Make sure you're running this notebook from the Notebooks/ directory.")

## Download Genome Data

Download M. tuberculosis genome data for H37Rv (reference strain) and CDC1551 (clinical strain) from NCBI.

In [ ]:
# Initialize the downloader
downloader = MTBDataDownloader(project_root)
print("Starting data download from NCBI...")
results = downloader.download_all_genomes()

# Display download summary
df_summary = pd.DataFrame(results).T
display(df_summary)

## Verify Downloaded Files

Check that all required files are present and validate their integrity.

In [ ]:
def verify_downloaded_files(strain):
    """Verify and display information about downloaded files."""
    raw_dir = project_root / "Data" / "Raw"

    print(f"n{'='*40}")
    print(f"Verifying {strain} files:")
    print(f'{'='*40}')

    files = {
        'FASTA': raw_dir / f"{strain}_fasta.fna",
        'GBFF': raw_dir / f"{strain}_gbff.gbff",
        'GFF3': raw_dir / f"{strain}_gff.gff",
    }

    for file_type, file_path in files.items():
        if file_path.exists():
            file_size = file_path.stat().st_size / (1024**2)
            print(f"✓ {file_type}: {file_path.name} ({file_size:.2f} MB)")
        else:
            print(f"✗ {file_type}: File not found")
    return all(file_path.exists() for file_path in files.values())

# Verify both strains
for strain in ['H37Rv', 'CDC1551']:
    verify_downloaded_files(strain)

## Explore FASTA Files

Examine basic characteristics of the genome sequences.

In [ ]:
from Bio import SeqIO

def explore_fasta_file(strain):
    """Explore basic characteristics of FASTA file."""
    fasta_path = project_root / "Data" / "Raw" / f"{strain}_fasta.fna"

    if not fasta_path.exists():
        print(f"FASTA file not found for {strain} .")
        return None
    
    records = list(SeqIO.parse(fasta_path, "fasta"))

    print(f"n{strain} Genome Summary:")
    print(f"{'='*40}")
    print(f"Number of contigs: {len(records)}")

    contig_lengths = [len(record.seq) for record in records]
    total_length = sum(contig_lengths)
    print(f"Total length: {total_length:,} bp")
    print(f"Contig lengths (top 3): {[f'{l:,}' for l in sorted(contig_lengths, reverse=True)[:3]]}...")

    # Calculate GC content with workaround function
    gc_total = 0
    for rec in records:
        gc_total += gc_fraction(rec.seq) * len(rec.seq)
    gc_content = (gc_total / total_length) * 100
    print(f"GC Content: {gc_content:.2f}%")

    # Display first contig info
    print(f"\\nFirst Contig ({records[0].id}):")
    print(f"Length: {len(records[0].seq):,} bp")
    print(f"First 50 bases: {records[0].seq[:50]}")
    print(f"Last 50 bases: {records[0].seq[-50:]}")

    return records

# Explore both genomes
h37rv_records = explore_fasta_file("H37Rv")
cdc1551_records = explore_fasta_file("CDC1551")

## QC Checks

Perform comprehensive quality control on the downloaded data.

In [ ]:
def perform_qc_checks(strain):
    """Perform quality control checks on genome data."""
    print(f"\\nQuality Control for {strain}:")
    print(f"{'='*40}")

    raw_dir = project_root / "Data" / "Raw"
    checks = []

    # Check 1: File existence
    required_files = ['_fasta.fna', '_gbff.gbff', '_gff.gff']
    for suffix in required_files:
        file_path = raw_dir / f"{strain}{suffix}"
        if file_path.exists():
            checks.append(("File exists: " + suffix[1:], file_path.exists()))

    # Check 2: Basic FASTA integrity
    fasta_path = raw_dir / f"{strain}_fasta.fna"
    if fasta_path.exists():
        try:
            records = list(SeqIO.parse(fasta_path, "fasta"))
            checks.append(("FASTA parses successfully", len(records) > 0))
            if len(records) > 0:
                checks.append(("Contig count reasonable", 1 <= len(records) <= 10))
                total_length = sum(len(rec.seq) for rec in records)
                checks.append(("Genome length reasonable", 4e6 <= total_length <= 5e6))
        except Exception as e:
            checks.append(("Fasta parsing", False))
    
    # Check 3: GFF file has content
    gff_path = raw_dir / f"{strain}_gff.gff"
    if gff_path.exists():
        try:
            with open(gff_path, 'r') as f:
                lines = f.readlines()
                gene_lines = [l for l in lines if 'gene' in l or 'CDS' in l]
                checks.append(("GFF file has gene annotations", len(gene_lines) > 1000))
        except Exception as e:
            checks.append(("GFF reading", False))
    
    # Display QC results
    for check, passed in checks:
        status = "✓" if passed else "✗"
        print(f"{status} {check}")
    
    return all(passed for _, passed in checks)

# Perform QC checks for both strains
print("\\n" + "="*60)
print("Quality Control Summary")
print("="*60)

qc_results = {}
for strain in ['H37Rv', 'CDC1551']:
    qc_results[strain] = perform_qc_checks(strain)

print(f"\\nOverall QC Status:")
for strain, passed in qc_results.items():
    status = "PASS" if passed else "FAIL"
    print(f"{strain}: {status}")

## Analyze Genomes Using GenomeAnalyzer

Use the GenomeAnalyzer module to get detailed statistics about each genome.

In [ ]:
print("Initializing GenomeAnalyzer...")
analyzer = GenomeAnalyzer(project_root)

# Analyze both strains
for strain in ['H37Rv', 'CDC1551']:
    print(f"\\n{'=' * 40}")
    print(f"Detailed Analysis: {strain}")
    print(f"{'=' * 40}")
    try:
        records = analyzer.load_genome(strain)
        stats = analyzer.calculate_basic_stats(records)

        #Display key statistics
        print(f"Total Length: {stats['total_length']:,} bp")
        print(f"GC content: {stats['gc_content']:.2f}%")
        print(f"Number of Contigs: {stats['num_contigs']}")
        print(f"N50: {stats['n50']:,} bp")
        print(f"N75: {stats['n75']:,} bp")
        print(f"L50: {stats['l50']}")
        print(f"Contig length range: {stats['min_contig_length']:,} bp - {stats['max_contig_length']:,} bp")

    except Exception as e:
        print(f"Error analyzing {strain}: {e}")

## Compare Genomes

Compare the two strains to identify differences in genome characteristics.

In [ ]:
print(f"\\nH37Rv vs CDC1551 Comparison:")
print(f"{'='*40}")
print(f"Length difference: {comparison['length_difference']:,} bp")
print(f"Length ratio (H37Rv/CDC1551): {comparison['length_ratio']:.4f}")
print(f"GC content difference: {comparison['gc_difference']:.2f}%")
print(f"Contig count difference: {comparison['contig_difference']}")

# Access Individual statistics
h37rv_stats = comparison['stats1']
cdc_stats = comparison['stats2']

print(f"\\nH37Rv Statistics:")
print(f" Total Length: {h37rv_stats['total_length']:,} bp")
print(f" GC content: {h37rv_stats['gc_content']:.2f}%")

print(f"\\nCDC1551 Statistics:")
print(f" Total Length: {cdc_stats['total_length']:,} bp")
print(f" GC content: {cdc_stats['gc_content']:.2f}%")

## Create Visualizations

Generate visual comparisons of genome statistics.

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Load stats for both strains
h37rv_records = analyzer.load_genome("H37Rv")
cdc1551_records = analyzer.load_genome("CDC1551")

h37rv_stats = analyzer.calculate_basic_stats(h37rv_records)
cdc1551_stats = analyzer.calculate_basic_stats(cdc1551_records)

# Plot 1: Contig length comparison
strains = ['H37Rv', 'CDC1551']
lengths = [h37rv_stats['total_length'], cdc1551_stats['total_length']]

axes[0, 0].bar(strains, lengths, color=['skyblue', 'lightgreen'])
axes[0, 0].set_ylabel('Genome Length (bp)')
axes[0, 0].set_title('Total Genome Length')
axes[0, 0].ticklabel_format(style='sci', axis='y', scilimits=(6,6))

# Add value labels to plot
for i, v in enumerate(lengths):
    axes[0, 0].text(i, v, f"{v:,}", ha='center', va='bottom')

# Plot 2: GC content comparison
gc_contents = [h37rv_stats['gc_content'], cdc_stats['gc_content']]
axes[0, 1].bar(strains, gc_contents, color=['skyblue', 'lightgreen'])
axes[0, 1].set_ylabel('GC Content (%)')
axes[0, 1].set_title('GC Content Comparison')

# Add value labels to plot
for i, v in enumerate(gc_contents):
    axes[0, 1].text(i, v, f"{v:.2f}%", ha='center', va='bottom')

# Plot 3: Contig length distribution for H37Rv
contig_lengths_h37rv = h37rv_stats['contig_lengths']
axes[1, 0].hist(contig_lengths_h37rv, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Contig Length (bp)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('H37Rv Contig Length Distribution')
axes[1, 0].ticklabel_format(style='sci', axis='x', scilimits=(6,6))

# Plot 4: Contig length distribution for CDC1551
contig_lengths_cdc = cdc_stats['contig_lengths']
axes[1, 1].hist(contig_lengths_cdc, bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Contig Length (bp)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('CDC1551 Contig Length Distribution')
axes[1, 1].ticklabel_format(style='sci', axis='x', scilimits=(6,6))
plt.suptitle('Mycobacterium tuberculosis Strain Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save the figure
output_dir = project_root / "Results" / "Plots"
output_dir.mkdir(parents=True, exist_ok=True)
plt.save(output_dir / "genome_statistics_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Visualization saved to: {output_dir / 'genome_statistics_comparison.png'}")

## Generate Summary Report

Creates a comprehensive summary of the data aquisition and process.

In [ ]:
print("\\n" + "="*60)
print("DATA ACQUISITION SUMMARY")
print("="*60)

# Create summary table
summary_data = []

for strain in ['H37Rv', 'CDC1551']:
    # Get file information
    raw_dir = project_root / "Data" / "Raw"
    files = {
        'FASTA': raw_dir / f"{strain}_fasta.fna",
        'GBFF': raw_dir / f"{strain}_gbff.gbff",
        'GFF3': raw_dir / f"{strain}_gff.gff",
    }
    # Calculate statistics if available
    try:
        records = analyzer.load_genome(strain)
        stats = analyzer.calculate_basic_stats(records)

        summary_data.append({
            'Strain': strain,
            'Total Length (bp)': f"{stats['total_length']:,}",
            'GC Content (%)': f"{stats['gc_content']:.2f}",
            'Num Contigs': stats['num_contigs'],
            'N50 (bp)': f"{stats['n50']:,}",
            'Files Downloaded': sum(1 for f in files.values() if f.exists())
            'Status': 'Complete' if all(f.exists() for f in files.values()) else 'Incomplete'
        })
    except Exception as e:
        summary_data.append({
            'Strain': strain,
            'Total Length (bp)': 'N/A',
            'GC Content (%)': 'N/A',
            'Num Contigs': 'N/A',
            'N50 (bp)': 'N/A',
            'Files Downloaded': sum(1 for f in files.values() if f.exists())
            'Status': 'Error'
        })

# Display summary table
summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Save summary to file
output_file = project_root / "Results" / "data_ascuisition_summary.csv"
summary_df.to_csv(output_file, index=False)
print(f"\\n✓ Summary saved to: {output_file}")
print(f"\\nData acquisition and quality control complete!")